# Monitoring variant: ocudu-zmq/zmq-split/split-ran.ipynb

Based on the upstream example. The original remains under `examples/ocudu-zmq/split-ran.ipynb`.
Core: **open5gs**; RAN: **ocudu**. Open5GS slice KPIs use the custom SMF/UPF image. The OAI monolithic-gNB KPI calculator is disabled for this RAN layout.

# **5G Deployment Example**

In this example, we will deploy a 5G network with:

- Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR
- OCUDU CU-CP, CU-UP, and DU with ZeroMQ
- SRSUE with ZeroMQ as simulated UE

Reference: https://docs.ocudu.org/tutorials/srsue/#zeromq-based-setup

First, we install the required dependencies:

## Install Dependencies:

Install the SLICES CLI used for the BI to create VMs, and for POS to provision baremetal serevers:

In [ ]:
%pip install slices-cli --extra-index-url=https://doc.slices-ri.eu/pypi/

Install the SLICES POS CLI:

In [ ]:
%pip install slices-cli-pos --index-url https://gitlab.inria.fr/api/v4/projects/68732/packages/pypi/simple --extra-index-url https://doc.slices-ri.eu/pypi/

Install the Post-5G orchestrator:

In [ ]:
from pathlib import Path
import os
import sys

REPOSITORY = Path("../../../..").resolve()
# Refresh upstream even when its package version number has not changed.
%pip install --upgrade --force-reinstall --no-deps "orchestrator @ git+https://gitlab.inria.fr/slices-ri/blueprints/post-5g/orchestrator.git@main"
%pip install -e "{REPOSITORY}" --extra-index-url=https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]
sys.path.insert(0, str(REPOSITORY / "src"))


## Input 5G Network Description:

In [ ]:
from metamodel.models.metamodel import MetaModel

metamodel = MetaModel.from_yaml(
"""
rus:
  sim_ru_du:
    txGainDB: 0
    rxGainDB: 0

  sim_ru_ue:
    txGainDB: 50
    rxGainDB: 40

ran:
  components:
    cucp:
      type: cu_cp
    cuup:
      type: cu_up
      config:
        cu_cp: cucp
    du:
      type: du
      config:
        ru: sim_ru_du
        cu_cp: cucp
        cu_up: cuup 
        radioConfiguration:
          nrBand: 3
          downlinkArfcn: 368500
          channelBandwidthMhz: 10
          subcarrierSpacingKhz: 15
          ssbPeriodicityMs: 10
          prachConfigurationIndex: 1

ues:
  sim_ue:
    config:
      nrBands: [3]
      ru: sim_ru_ue
      gnbs: [du]

slices:
  internet:
    sst: 1
    sd: 0xFFFFFF # explicitly set sd to avoid core regection
"""
)

In [ ]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
params:
  provisioning:
    project: post5g-beta
    experiment: 5gbp_exp
    flavor: xlarge # ocudu zmq needs more than 5 CPU cores (see: https://github.com/srsran/srsRAN_Project/issues/1472)
    extra_path: "" # Use SLICES CLI from PATH

ran:
  implementation: ocudu
  components:
    cucp:
      type: cu_cp
      advanced_config:
        inactivity_timer: 7200
    du:
      type: du
      advanced_config:
        srate: 11.52
        base_srate: "11.52e6"
        pdcch:
          common:
            ss0_index: 0
            coreset0_index: 6
          dedicated:
            ss2_type: common
            dci_format_0_1_and_1_1: false
        pdsch:
          mcs_table: qam64
        pusch:
          mcs_table: qam64

ues:
  sim_ue:
    type: containerized
    implementation: srsue
    advanced_config:
      srate: "11.52e6"
"""
)

## Provisioning:

In [ ]:
from orchestrator.deployment import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [ ]:
await prov.provision()

## Kubernetes Deployment:

In [ ]:
from orchestrator.deployment import K8SManagement

k8s = K8SManagement(prov)

In [ ]:
await k8s.deploy()

## 5G Core Deployment:

In [ ]:
from orchestrator.deployment import CoreManagement

core = CoreManagement(k8s)

In [ ]:
# Configure the Open5GS metrics image before preparing its built-in charts.
from importlib.resources import files
from orchestrator.models.core import ADVANCED_CORE_CONFIG_MAPPING
from orchestrator.models.k8s import ContainerImage

metrics_templates = files("fiveg_lens") / "assets" / "core" / "open5gs-metrics"

for component in core.deployment_model.core.components.values():
    if component.implementation != "open5gs" or component.type not in ("smf", "upf"):
        continue
    component.image = ContainerImage(
        repository="ghcr.io/niloysh/open5gs",
        image_tag="v2.7.0-upf-metrics-v2",
    )
    advanced = component.advanced_config.model_dump() if component.advanced_config else {}
    advanced["logger"] = {"file": f"/tmp/open5gs-{component.type}.log", "level": "info"}
    if component.type == "upf":
        advanced.setdefault("teid_range_indication", 0)
    config_type = ADVANCED_CORE_CONFIG_MAPPING[component.implementation][component.type]
    component.advanced_config = config_type.model_validate(advanced)
    component.custom_rendered_files = [
        str(metrics_templates / "values.yaml.j2"),
        str(metrics_templates / "config.yaml.j2"),
    ]
    if component.start and not component.custom_entrypoint_command:
        component.custom_entrypoint_command = [
            f"/open5gs/install/bin/open5gs-{component.type}d",
            "-c",
            "/opt/open5gs/etc/open5gs/config.yaml",
        ]


In [ ]:
await core.run_database_action(action="prepare")
await core.prepare()


In [ ]:
# Preserve the database and subscriber steps performed by upstream core.deploy().
await core.run_database_action(action="install")
await core.install()
await core.register_ues()


## RAN Deployment:

In [ ]:
from orchestrator.deployment import RANManagement

ran = RANManagement(core)

In [ ]:
await ran.deploy()

Get OCUDU ZMQ DU logs:

In [ ]:
print(await ran.get_logs(quiet=True))

## UE Deployement:

## Monitoring before UE attachment

The profile uses the control host selected by Kubernetes, including physical
Sopnodes. Capture interfaces refer to pod interfaces, not the host or radio NIC.
Keep the existing provisioning, RF and Multus settings. Grafana uses `admin` / `admin`.

In [ ]:
from pathlib import Path
import asyncio
import json

from fiveg_lens import MonitoringManagement
from fiveg_lens.models import MonitoringDeployment

QUERIES_FILE = Path("queries.json")  # Your Prometheus export query profile.

monitoring = MonitoringManagement(
    preceding=ran,
    config=MonitoringDeployment.from_yaml_file("monitoring.yaml"),
)


In [ ]:
try:
    await monitoring.deploy()
finally:
    monitoring.output(
        metamodel_file="monitoring_metamodel.yaml",
        deployment_model_file="monitoring_deployment_model.yaml",
    )

### Open Grafana

The SSH tunnel uses the current model address and jump host.
Open the link after deployment; close the tunnel with `grafana_tunnel.terminate()`.

In [ ]:
import atexit
import shlex
import subprocess
from IPython.display import Markdown, display

# Close the previous tunnel before opening one with the current host address.
if "grafana_tunnel" in globals() and grafana_tunnel.poll() is None:
    grafana_tunnel.terminate()
    await asyncio.to_thread(grafana_tunnel.wait, 5)

# get SSH info and fallback
cfg = monitoring.deployment_model.monitoring
control = monitoring.deployment_model.params.k8s.master_node
ssh = monitoring.deployment_model.nodes[control].ssh_info

command = ["ssh", "-N", "-T", "-S", "none", "-o", "ControlMaster=no", "-o", "ControlPersist=no",
           "-o", "BatchMode=yes", "-o", "ExitOnForwardFailure=yes", "-o", "ConnectTimeout=10"]

command += shlex.split(ssh.common_args or "")

if ssh.private_key_file:
    command += ["-i", str(Path(ssh.private_key_file).expanduser())]
command += ["-p", str(ssh.port), "-L", f"127.0.0.1:30096:127.0.0.1:{cfg.grafana_node_port}",
            f"{ssh.user}@{ssh.hostname or control}"]

grafana_tunnel = subprocess.Popen(command, stdout=subprocess.DEVNULL)

# Close the tunnel when the kernel exits normally.
atexit.register(grafana_tunnel.terminate)

GRAFANA_URL = "http://127.0.0.1:30096"
display(Markdown(f"[Open Grafana]({GRAFANA_URL}) — username: **admin**, password: **admin**"))

In [ ]:
from orchestrator.deployment.ue import UEManagement

ue = UEManagement(preceding=ran)

In [ ]:
await ue.deploy()

Get SRSUE ZMQ UE logs:

In [ ]:
print(await ue.get_logs(quiet=True))

Get DU logs after UE was connected:

In [ ]:
print(await ran.get_logs(quiet=True))

Verify that the UE got a PDU session:

In [ ]:
print((await ue.run_command(command=["ip", "addr"], quiet=True, flatten=False))["sim_ue"])


#### Test connectivity:

In [ ]:
print((await ue.run_command(command=["apt", "update"], quiet=True, flatten=False))["sim_ue"])


In [ ]:
print((await ue.run_command(command=["apt", "install", "-y", "iputils-ping"], quiet=True, flatten=False))["sim_ue"])


In [ ]:
print(
    (await ue.run_command(command=["ping", "11.1.1.1", "-c", "6", "-I", "tun_srsue"], quiet=True, flatten=False))[
        "sim_ue"
    ]
)


In [ ]:
print(
    (await ue.run_command(command=["ping", "8.8.8.8", "-c", "6", "-I", "tun_srsue"], quiet=True, flatten=False))[
        "sim_ue"
    ]
)


## Test another split: Change RAN split from E1-split to F1-split

We can change the OCUDU RAN's split from E1 to F1 while keeping the already deployed Open5GS core.

To do so, remove the deployed UE and RAN:

In [ ]:
await monitoring.remove()
await ue.remove(quiet=True)
await ran.remove(quiet=True)

Remove existing OCUDU E1s-split RAN components:

In [ ]:
ran.remove_component("cuup")
ran.remove_component("cucp")
ran.remove_component("du")

Add CU that will replace the previous CU-CP and CU-UP:

In [ ]:
from metamodel.models.ran import CUComponent, RANComponentType
from orchestrator.models.ran import DeployedRANComponent

ran.add_component(
    name="cu",
    component_meta=CUComponent(type=RANComponentType.CU),
    component_deployment=DeployedRANComponent.from_yaml(
"""
type: cu
implementation: ocudu
advanced_config:
  inactivity_timer: 7200
"""
    ),
)

Update and add the F1-split RAN DU from the previous one:

In [ ]:
du_meta = metamodel.ran.components["du"]
du_deployemnt = deployment_model.ran.components["du"]

# update cu_cp and cu_up references in du component
du_meta.config.cu_cp = "cu"
du_meta.config.cu_up = "cu"

ran.add_component(
    name="du",
    component_meta=du_meta,
    component_deployment=du_deployemnt,
)

Deploy the OCUDU F1-split RAN:

In [ ]:
await ran.deploy()

In [ ]:
# The second layout replaces CU-CP/CU-UP with a combined CU.
monitoring = MonitoringManagement(
    preceding=ran,
    config=MonitoringDeployment.from_yaml_file("monitoring.yaml"),
)
packet_target = monitoring.deployment_model.monitoring.targets[-1]
packet_target.component = "cu"
packet_target.container = "cu"
await monitoring.deploy()


Redeploy the SRSUE based on the changed RAN:

In [ ]:
ue = UEManagement(preceding=ran)

In [ ]:
await ue.deploy()

Get DU logs after UE was connected:

In [ ]:
print(await ran.get_logs(quiet=True))

Verify that the UE got a PDU session:

In [ ]:
print((await ue.run_command(command=["ip", "addr"], quiet=True, flatten=False))["sim_ue"])


#### Test connectivity:

In [ ]:
print((await ue.run_command(command=["apt", "update"], quiet=True, flatten=False))["sim_ue"])


In [ ]:
print((await ue.run_command(command=["apt", "install", "-y", "iputils-ping"], quiet=True, flatten=False))["sim_ue"])


In [ ]:
print(
    (await ue.run_command(command=["ping", "11.1.1.1", "-c", "6", "-I", "tun_srsue"], quiet=True, flatten=False))[
        "sim_ue"
    ]
)


In [ ]:
print(
    (await ue.run_command(command=["ping", "8.8.8.8", "-c", "6", "-I", "tun_srsue"], quiet=True, flatten=False))[
        "sim_ue"
    ]
)


## Iperf with experiment artifacts

The server runs on the Kubernetes control host from the deployment model.
Each client binds to its own 5G interface. Review the explicit scenario before running it.
The collector exports Prometheus data after traffic and saves each report with the timeline.

In [ ]:
from datetime import UTC, datetime
from fiveg_lens.artifacts import ExperimentRun
from orchestrator.deployment import ProvisioningManagement
from orchestrator.utils.ansible import get_ansible_inventory_line, run_ansible_command, run_ansible_playbook

SERVER_NODE = monitoring.deployment_model.params.k8s.master_node
CLIENTS = ["sim_ue"]
UE_INTERFACE = "tun_srsue"
DURATION, BASE_PORT = 20, 5202
BASELINE_SECONDS, COLLECTION_SECONDS = 5, 5

server_node = monitoring.deployment_model.nodes[SERVER_NODE]
server_ip = server_node.ssh_info.hostname or SERVER_NODE
server_inventory = get_ansible_inventory_line(SERVER_NODE, server_node)
server_runner = str(Path(monitoring.deployment_model.params.ansible.runner_path) / "iperf-server")
clients = ue

await run_ansible_playbook(
    playbook=[
        {
            "name": "Install iperf3 on the server host",
            "hosts": SERVER_NODE,
            "gather_facts": False,
            "become": True,
            "tasks": [
                {
                    "ansible.builtin.apt": {
                        "name": "iperf3",
                        "state": "present",
                        "update_cache": True,
                        "cache_valid_time": 3600,
                    }
                }
            ],
        }
    ],
    inventory=server_inventory,
    runner_path=server_runner,
    quiet=True,
)
await clients.run_command(command=["apt-get", "update"], components=CLIENTS, quiet=True, flatten=False)
await clients.run_command(
    command=["env", "DEBIAN_FRONTEND=noninteractive", "apt-get", "install", "-y", "iperf3"],
    components=CLIENTS,
    quiet=True,
    flatten=False,
)
addresses = await clients.run_command(
    command=["ip", "-j", "-4", "addr", "show", "dev", UE_INTERFACE], components=CLIENTS, quiet=True, flatten=False
)
ue_ips = {}
for client in CLIENTS:
    ipv4 = [
        a["local"] for device in json.loads(addresses[client]) for a in device["addr_info"] if a["family"] == "inet"
    ]
    if len(ipv4) != 1:
        raise RuntimeError(f"{client}: expected one IPv4 address on {UE_INTERFACE}")
    ue_ips[client] = ipv4[0]
print(f"Server: {SERVER_NODE} ({server_ip}); UE clients: {ue_ips}")


In [ ]:
async def start_iperf_server(port):
    await run_ansible_command(
        command=["iperf3", "-s", "-D", "-1", "--idle-timeout", "60", "-B", server_ip, "-p", str(port), "-J"],
        hosts=SERVER_NODE,
        inventory=server_inventory,
        runner_path=server_runner,
        quiet=True,
    )


async def iperf_client(client, direction, duration, port):
    reverse = {"uplink": [], "downlink": ["-R"]}[direction]
    output = await clients.run_command(
        command=[
            "iperf3",
            "-c",
            server_ip,
            "-B",
            ue_ips[client],
            "--bind-dev",
            UE_INTERFACE,
            "-p",
            str(port),
            "-t",
            str(duration),
            "--connect-timeout",
            "10000",
            "-J",
            "--get-server-output",
            *reverse,
        ],
        components=[client],
        quiet=True,
        flatten=False,
    )
    report = json.loads(output[client])
    if report.get("error"):
        raise RuntimeError(report["error"])
    return report


In [ ]:
SCENARIO = [
    {"name": "ue1_uplink", "client": 'sim_ue', "direction": "uplink", "duration": DURATION},
    {"name": "ue1_downlink", "client": 'sim_ue', "direction": "downlink", "duration": DURATION},
]

In [ ]:
timestamp = datetime.now(UTC).strftime("%Y-%m-%d_%H-%M-%S.%fZ")
output_directory = Path("results") / f"iperf-{timestamp}"

async with ExperimentRun(
    output_directory,
    name="ue-to-host-iperf",
    monitoring=monitoring,
    queries_file=QUERIES_FILE,
) as experiment:
    experiment.save_json("scenario.json", SCENARIO)

    with experiment.section("baseline"):
        await asyncio.sleep(BASELINE_SECONDS)

    for port, test in enumerate(SCENARIO, start=BASE_PORT):
        print(f"Running {test['name']}: {test['client']}, {test['direction']}, {test['duration']} seconds")
        await start_iperf_server(port)

        with experiment.section(test["name"]):
            report = await iperf_client(
                client=test["client"],
                direction=test["direction"],
                duration=test["duration"],
                port=port,
            )

        experiment.save_json(f"{test['name']}.json", report)

    with experiment.section("collection"):
        await asyncio.sleep(COLLECTION_SECONDS)

print(f"Artifacts saved to: {output_directory.resolve()}")